<a href="https://colab.research.google.com/github/Fabi-Aco/DesktopBot/blob/main/EPDS_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import re
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Safely check for matching files in the directory
matched_files = [
    f
    for f in os.listdir(".")
    if "India_Peak_Load" in f or f.endswith(".xlsx")
]

matches = []
if matched_files:
    file_name = matched_files[0]
    with open(file_name, "rb") as f:
        raw_text = f.read().decode("utf-8", errors="ignore")
    pattern = r"(20\d{2}-\d{2})[^\d\n]*?(\d+)[^\d\n\-]*?(-?\d+)[^\d\n]*?(\d+\.\d+|\d+)"
    matches = re.findall(pattern, raw_text)

# If no file was found or no matches extracted, use fallback data
if not matches:
    data = [
        ["2016-17", 1, -9, 160.18],
        ["2017-18", 2, -7, 164.06],
        ["2018-19", 3, -5, 177.02],
        ["2019-20", 4, -3, 183.59],
        ["2020-21", 5, -1, 190.03],
        ["2021-22", 6, 1, 203.01],
        ["2022-23", 7, 3, 215.89],
        ["2023-24", 8, 5, 243.27],
        ["2024-25", 9, 7, 250.00],
        ["2025-26", 10, 9, 265.00],
    ]
    df = pd.DataFrame(
        data,
        columns=["Financial_Year", "Time_Index", "Coded_Time", "Peak_Demand_GW"],
    )
else:
    df = pd.DataFrame(
        matches,
        columns=["Financial_Year", "Time_Index", "Coded_Time", "Peak_Demand_GW"],
    )
    df["Time_Index"] = df["Time_Index"].astype(int)
    df["Coded_Time"] = df["Coded_Time"].astype(int)
    df["Peak_Demand_GW"] = df["Peak_Demand_GW"].astype(float)
    df = (
        df.drop_duplicates(subset=["Time_Index"])
        .sort_values("Time_Index")
        .reset_index(drop=True)
    )

df.to_csv("clean_peak_load.csv", index=False)
print(df)

# Reload data for modeling
df = pd.read_csv("clean_peak_load.csv")

X = df[["Time_Index"]]  # Time Index: t = 1 to 10
y = df["Peak_Demand_GW"]

baseline_model = LinearRegression()
baseline_model.fit(X, y)

  Financial_Year  Time_Index  Coded_Time  Peak_Demand_GW
0        2016-17           1          -9          160.18
1        2017-18           2          -7          164.06
2        2018-19           3          -5          177.02
3        2019-20           4          -3          183.59
4        2020-21           5          -1          190.03
5        2021-22           6           1          203.01
6        2022-23           7           3          215.89
7        2023-24           8           5          243.27
8        2024-25           9           7          250.00
9        2025-26          10           9          265.00


LinearRegression()

In [4]:
y_pred_baseline = baseline_model.predict(X)

next_year_df = pd.DataFrame({"Time_Index": [13]}) #Put Time Index to predict
baseline_pred_next = baseline_model.predict(next_year_df)[0]

In [5]:
rmse_baseline = np.sqrt(mean_squared_error(y, y_pred_baseline))
r2_baseline = r2_score(y, y_pred_baseline)

In [6]:
print(
    f"Linear Equation: y = {baseline_model.intercept_:.2f} +"
    f" {baseline_model.coef_[0]:.2f} * t"
)
print(f"R² Score:        {r2_baseline:.4f}")
print(f"RMSE:            {rmse_baseline:.2f} GW")
print(
    f"\nPredicted Peak Load: {baseline_pred_next:.2f} GW\n"
)

Linear Equation: y = 139.00 + 12.04 * t
R² Score:        0.9669
RMSE:            6.39 GW

Predicted Peak Load: 295.48 GW



In [16]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Hides info and warning logs from TensorFlow
import re
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

#random seeds for LSTM results

tf.random.set_seed(42)
np.random.seed(42)

# 1. DATA LOADING & PREPROCESSING

matched_files = [
    f
    for f in os.listdir(".")
    if "India_Peak_Load" in f or f.endswith(".xlsx")
]

matches = []
if matched_files:
    file_name = matched_files[0]
    with open(file_name, "rb") as f:
        raw_text = f.read().decode("utf-8", errors="ignore")
    pattern = r"(20\d{2}-\d{2})[^\d\n]*?(\d+)[^\d\n\-]*?(-?\d+)[^\d\n]*?(\d+\.\d+|\d+)"
    matches = re.findall(pattern, raw_text)

if not matches:
    data = [
        ["2016-17", 1, -9, 160.18],
        ["2017-18", 2, -7, 164.06],
        ["2018-19", 3, -5, 177.02],
        ["2019-20", 4, -3, 183.59],
        ["2020-21", 5, -1, 190.03],
        ["2021-22", 6, 1, 203.01],
        ["2022-23", 7, 3, 215.89],
        ["2023-24", 8, 5, 243.27],
        ["2024-25", 9, 7, 250.00],
        ["2025-26", 10, 9, 265.00],
    ]
    df = pd.DataFrame(
        data,
        columns=["Financial_Year", "Time_Index", "Coded_Time", "Peak_Demand_GW"],
    )
else:
    df = pd.DataFrame(
        matches,
        columns=["Financial_Year", "Time_Index", "Coded_Time", "Peak_Demand_GW"],
    )
    df["Time_Index"] = df["Time_Index"].astype(int)
    df["Coded_Time"] = df["Coded_Time"].astype(int)
    df["Peak_Demand_GW"] = df["Peak_Demand_GW"].astype(float)
    df = (
        df.drop_duplicates(subset=["Time_Index"])
        .sort_values("Time_Index")
        .reset_index(drop=True)
    )

df.to_csv("clean_peak_load.csv", index=False)
df = pd.read_csv("clean_peak_load.csv")

X_time = df["Time_Index"].values
y_demand = df["Peak_Demand_GW"].values

# 2. BASELINE LINEAR REGRESSION

linear_model = LinearRegression()
linear_model.fit(X_time.reshape(-1, 1), y_demand)
y_pred_linear = linear_model.predict(X_time.reshape(-1, 1))

rmse_linear = np.sqrt(mean_squared_error(y_demand, y_pred_linear))
r2_linear = r2_score(y_demand, y_pred_linear)

print("--- Linear Regression Model ---")
print(f"RMSE: {rmse_linear:.4f} GW")
print(f"R2 Score: {r2_linear:.4f}\n")

# 3. LEAST SQUARES PARABOLIC METHOD

quad_coeffs = np.polyfit(X_time, y_demand, 2)
quad_model = np.poly1d(quad_coeffs)
y_pred_quad = quad_model(X_time)

rmse_quad = np.sqrt(mean_squared_error(y_demand, y_pred_quad))
r2_quad = r2_score(y_demand, y_pred_quad)

print("--- Least Squares Parabolic Model ---")
print(f"Equation: y = {quad_coeffs[0]:.4f}x^2 + {quad_coeffs[1]:.4f}x + {quad_coeffs[2]:.4f}")
print(f"RMSE: {rmse_quad:.4f} GW")
print(f"R2 Score: {r2_quad:.4f}\n")

# 4. LSTM NEURAL NETWORK MODEL
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['Peak_Demand_GW']])

def create_sequences(data, look_back=2):
    X_seq, y_seq = [], []
    for i in range(len(data) - look_back):
        X_seq.append(data[i:(i + look_back), 0])
        y_seq.append(data[i + look_back, 0])
    return np.array(X_seq), np.array(y_seq)

look_back = 2
X_lstm, y_lstm = create_sequences(scaled_data, look_back)
X_lstm = np.reshape(X_lstm, (X_lstm.shape[0], X_lstm.shape[1], 1))

lstm_model = Sequential([
    Input(shape=(look_back, 1)),
    LSTM(50, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.fit(X_lstm, y_lstm, epochs=300, verbose=0)

train_predict_scaled = lstm_model.predict(X_lstm, verbose=0)
train_predict = scaler.inverse_transform(train_predict_scaled)
y_lstm_actual = scaler.inverse_transform(y_lstm.reshape(-1, 1))

rmse_lstm = np.sqrt(mean_squared_error(y_lstm_actual, train_predict))
r2_lstm = r2_score(y_lstm_actual, train_predict)

print("--- LSTM Model ---")
print(f"RMSE: {rmse_lstm:.4f} GW")
print(f"R2 Score: {r2_lstm:.4f}")

--- Linear Regression Model ---
RMSE: 6.3939 GW
R2 Score: 0.9669

--- Least Squares Parabolic Model ---
Equation: y = 0.7133x^2 + 4.1907x + 154.6945
RMSE: 3.7440 GW
R2 Score: 0.9887

--- LSTM Model ---
RMSE: 5.6251 GW
R2 Score: 0.9672
